# Qwen-Image 2.1 NSFW / 3D Pose Control — Google Colab 完整版

本 Notebook 將 [**Arekejoker/qwen21-nsfw-pose-skill**](https://github.com/Arekejoker/qwen21-nsfw-pose-skill) 完整移植至 Google Colab 環境，支援原專案驗證過的 **Qwen-Image 2.1 (`int8 convrot`)** 底模、**NSFW LoRAs**、**136 個 3D 姿勢庫**與 **VNCCS 人偶精確姿勢控制**。

---

### 硬體資源需求與 Colab GPU 建議

底模（Diffusion 6.8GB + Text Encoder 8.7GB + VAE 0.6GB）與 LoRA 合計約 **16.7 GB ~ 18 GB**，建議在 Colab 選擇以下執行階段：

| Colab GPU 規格 | 顯存 (VRAM) | 系統記憶體 (RAM) | 相容性與表現 |
| :--- | :--- | :--- | :--- |
| **NVIDIA L4** *(強烈推薦)* | 22.5 ~ 24 GB | 53 ~ 62 GB | **最佳性價比**：模型可常駐顯存、不爆 RAM，50 步 (1152×1728) 約 2~3.5 分鐘/張 |
| **NVIDIA A100** | 40 / 80 GB | 83+ GB | **極速體驗**：單張約 40~90 秒 |
| **NVIDIA T4 (免費版)** | 15 GB | **12.7 GB** ⚠️ | **容易因系統 RAM 不足而崩潰 (OOM)**：若使用 T4，腳本會自動啟用 `--lowvram`，建議調降解析度至 `1024×1024` |

---

### 支援的四條生成路線

1. **路線 ① T2I（文字出圖 + NSFW LoRA）**：使用作者級 7 要素長提示詞 + `er_sde` / `beta` / CFG 2.5 / 50 步配方。
2. **路線 ② Edit（角色換姿勢/換裝）**：保留原圖臉部與特徵，透過提示詞修改姿勢或服裝。
3. **路線 ③ 腳本姿勢遷移（3D 人偶參考圖）**：輸入一張角色圖 + 一張無貼圖 3D 人偶渲染圖，免開瀏覽器完成精確姿勢遷移。
4. **路線 ④ VNCCS PoseStudio（136 個 3D 姿勢庫）**：透過內建 Headless Chromium (SwiftShader WebGL) 或 Cloudflare Tunnel 網頁介面操作 3D 人偶並出圖。

In [ ]:
#@title 1. 檢查 GPU、系統記憶體 (RAM) 與硬碟空間
import os
import shutil
import subprocess

print("=== GPU 資訊 ===")
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print(gpu_info.stdout.strip())
else:
    print("⚠️ 未偵測到 NVIDIA GPU！請至「執行階段 > 變更執行階段類型」選擇 L4 或 A100 GPU。")

# Check System RAM
mem_bytes = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')
mem_gb = mem_bytes / (1024.**3)
disk_total, disk_used, disk_free = shutil.disk_usage("/")
disk_free_gb = disk_free / (1024.**3)

print(f"\n=== 系統資源 ===")
print(f"系統記憶體 (System RAM): {mem_gb:.1f} GB")
print(f"可用硬碟空間 (Free Disk): {disk_free_gb:.1f} GB (建議至少 30 GB)")

if mem_gb < 20:
    print("\n⚠️ 注意：目前系統 RAM 小於 20GB（可能為免費版 T4）。")
    print("啟動 ComfyUI 時將自動加上 --lowvram 參數，且建議生成尺寸不超過 1024x1024 以免 CPU RAM OOM。")
else:
    print("\n✅ 系統記憶體充足，可流暢執行完整 1152x1728 配方！")

In [ ]:
#@title 2. 安裝 ComfyUI、qwen21-nsfw-pose-skill 與 VNCCS 節點 (~2-3 分鐘)
INSTALL_PLAYWRIGHT_FOR_ROUTE4 = True #@param {type:"boolean"}

import os
import subprocess

os.chdir("/content")

print("📦 [1/5] 安裝 aria2 高速下載工具...")
!apt-get update -qq && apt-get install -y -qq aria2

print("📦 [2/5] 下載並安裝 ComfyUI...")
if not os.path.exists("/content/ComfyUI"):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt

print("📦 [3/5] 下載 qwen21-nsfw-pose-skill 專案...")
if not os.path.exists("/content/qwen21-nsfw-pose-skill"):
    !git clone --depth 1 https://github.com/Arekejoker/qwen21-nsfw-pose-skill.git /content/qwen21-nsfw-pose-skill

print("📦 [4/5] 安裝 ComfyUI_VNCCS_Utils 自訂節點與依賴...")
vnccs_dir = "/content/ComfyUI/custom_nodes/ComfyUI_VNCCS_Utils"
if not os.path.exists(vnccs_dir):
    !git clone --depth 1 https://github.com/AHEKOT/ComfyUI_VNCCS_Utils.git {vnccs_dir}
!pip install -q pytorch-lightning roma "dynamicprompts>=0.31,<1" Pillow

if INSTALL_PLAYWRIGHT_FOR_ROUTE4:
    print("📦 [5/5] 安裝 Playwright 與 Headless Chromium（供路線 ④ 無頭 3D 渲染使用）...")
    !pip install -q playwright
    !playwright install chromium
    !playwright install-deps chromium
else:
    print("⏭️ [5/5] 略過 Playwright 安裝（路線 ①②③ 不需瀏覽器）。")

# 設定全域環境變數
os.environ["POSELAB"] = "/content/qwen21-nsfw-pose-skill"
os.environ["COMFY"] = "/content/ComfyUI"
os.environ["COMFYUI_URL"] = "http://127.0.0.1:8188"
os.environ["COMFYUI_INPUT"] = "/content/ComfyUI/input"
os.environ["COMFYUI_OUTPUT"] = "/content/ComfyUI/output"
os.environ["VNCCS_POSELIB"] = "/content/ComfyUI/custom_nodes/ComfyUI_VNCCS_Utils/PoseLibrary"

for sub in ["diffusion_models", "text_encoders", "vae", "loras"]:
    os.makedirs(f"/content/ComfyUI/models/{sub}", exist_ok=True)

print("\n✅ 環境與節點安裝完成！")

In [ ]:
#@title 3. 下載 Qwen-Image 2.1 底模 (3個檔案, ~16.1GB) 與 NSFW / VNCCS LoRAs
#@markdown > **Civitai API Token 說明**：路線 ① 的 NSFW LoRA（如 `tawsLoRa`、`TheseAlpacas`、`posture-v3`）託管於 Civitai。請填入你的 Civitai API Token（可於 [Civitai Account Settings](https://civitai.com/user/account) 底部產生），或在 Colab 左側「🔑 密鑰 (Secrets)」新增名為 `CIVITAI_TOKEN` 的密鑰。

CIVITAI_API_TOKEN = "" #@param {type:"string"}
HF_ENDPOINT = "https://huggingface.co" #@param ["https://huggingface.co", "https://hf-mirror.com"]
DOWNLOAD_DEFAULT_NSFW_LORA = True #@param {type:"boolean"}
DOWNLOAD_EXTRA_CIVITAI_LORAS = False #@param {type:"boolean"}
DOWNLOAD_EDIT_LORA = True #@param {type:"boolean"}
DOWNLOAD_VNCCS_POSE_LORA = True #@param {type:"boolean"}

import os
import subprocess

if not CIVITAI_API_TOKEN:
    try:
        from google.colab import userdata
        CIVITAI_API_TOKEN = userdata.get("CIVITAI_TOKEN") or ""
        if CIVITAI_API_TOKEN:
            print("🔑 已從 Colab Secrets 讀取 CIVITAI_TOKEN")
    except Exception:
        pass

def aria2_download(url: str, out_dir: str, out_name: str, token: str = ""):
    target_path = os.path.join(out_dir, out_name)
    if os.path.exists(target_path) and os.path.getsize(target_path) > 1024 * 1024:
        print(f"✔️ 已存在，略過下載: {out_name} ({os.path.getsize(target_path)/(1024**2):.1f} MB)")
        return
    print(f"⬇️ 正在下載: {out_name} ...")
    cmd = [
        "aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M",
        "-d", out_dir, "-o", out_name
    ]
    if token:
        # Civitai download supports token query param or Authorization header
        sep = "&" if "?" in url else "?"
        url_with_token = f"{url}{sep}token={token}"
        cmd.append(url_with_token)
    else:
        cmd.append(url)
    res = subprocess.run(cmd)
    if res.returncode != 0:
        print(f"❌ 下載失敗: {out_name}（請確認網路或 Civitai Token 是否正確）")
    else:
        print(f"✅ 完成: {out_name} ({os.path.getsize(target_path)/(1024**2):.1f} MB)")

comfy_models = "/content/ComfyUI/models"

# 1. 必備 3 個底模 (HF Comfy-Org/Qwen-Image-2.1)
base_models = [
    (
        f"{HF_ENDPOINT}/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
        f"{comfy_models}/diffusion_models",
        "qwen_image_2.1_int8_convrot.safetensors",
    ),
    (
        f"{HF_ENDPOINT}/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors",
        f"{comfy_models}/text_encoders",
        "qwen3vl_8b_int8_convrot.safetensors",
    ),
    (
        f"{HF_ENDPOINT}/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors",
        f"{comfy_models}/vae",
        "qwen_image_2.1_vae_bf16.safetensors",
    ),
]

for url, d, fn in base_models:
    aria2_download(url, d, fn)

# 2. HuggingFace LoRAs (路線 ② Edit & 路線 ③④ VNCCS)
if DOWNLOAD_EDIT_LORA:
    aria2_download(
        f"{HF_ENDPOINT}/RunningHubAI/rh-qwen-image-2.1-aio-nsfw-lora/resolve/main/Qwen-Image-2.1%20NSFW%20Image%20Edit.safetensors",
        f"{comfy_models}/loras",
        "qwen21-nsfw-image-edit.safetensors",
    )

if DOWNLOAD_VNCCS_POSE_LORA:
    aria2_download(
        f"{HF_ENDPOINT}/MIUProject/VNCCS_PoseStudio_QI2.1/resolve/main/VNCCS_QI2_PoseStudioV1.1.safetensors",
        f"{comfy_models}/loras",
        "VNCCS_QI2_PoseStudioV1.1.safetensors",
    )

# 3. Civitai NSFW LoRAs (路線 ①)
# Version IDs verified from Civitai API:
# - 3388157: model 2986844 (FemaledVaginus-NSFW v1.3b) -> qwen2_1-femaledVaginus-nsfw-v1_3-fine_L22-31.safetensors
# - 3386403: model 2986844 (FemaledVaginus-NSFW v1.3)  -> qwen2_1-femaledVaginus-nsfw-v1.3-fine.safetensors
# - 3357315: model 2958918 (TheseAlpacas v2.0)        -> thesealpacas-nsfw-v2.safetensors
# - 3388118: model 2988215 (nsfw-posture-v3 v1.0)     -> nsfw-posture-v3.safetensors
if DOWNLOAD_DEFAULT_NSFW_LORA:
    aria2_download(
        "https://civitai.com/api/download/models/3388157?type=Model&format=SafeTensor",
        f"{comfy_models}/loras",
        "qwen2_1-femaledVaginus-nsfw-v1_3-fine_L22-31.safetensors",
        token=CIVITAI_API_TOKEN,
    )

if DOWNLOAD_EXTRA_CIVITAI_LORAS:
    extra_civitai = [
        ("https://civitai.com/api/download/models/3386403?type=Model&format=SafeTensor", "qwen2_1-femaledVaginus-nsfw-v1.3-fine.safetensors"),
        ("https://civitai.com/api/download/models/3357315?type=Model&format=SafeTensor", "thesealpacas-nsfw-v2.safetensors"),
        ("https://civitai.com/api/download/models/3388118?type=Model&format=SafeTensor", "nsfw-posture-v3.safetensors"),
    ]
    for url, fn in extra_civitai:
        aria2_download(url, f"{comfy_models}/loras", fn, token=CIVITAI_API_TOKEN)

print("\n📂 目前已安裝的 LoRA 列表：")
!ls -lh /content/ComfyUI/models/loras

In [ ]:
#@title 4. 背景啟動 ComfyUI、同步 136 個 3D 姿勢庫並執行 `doctor.py` 自檢
import json
import os
import shutil
import subprocess
import time
import urllib.request

COMFY_URL = "http://127.0.0.1:8188"

def is_comfy_up() -> bool:
    try:
        with urllib.request.urlopen(f"{COMFY_URL}/system_stats", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if not is_comfy_up():
    mem_gb = (os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')) / (1024.**3)
    extra_args = ["--lowvram"] if mem_gb < 20 else []
    print(f"🚀 正在背景啟動 ComfyUI (extra_args={extra_args})...")
    log_file = open("/content/comfyui.log", "w")
    subprocess.Popen(
        ["python3", "main.py", "--listen", "127.0.0.1", "--port", "8188"] + extra_args,
        cwd="/content/ComfyUI",
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    for i in range(90):
        if is_comfy_up():
            print(f"✅ ComfyUI 已成功啟動於 {COMFY_URL}（耗時 {i*2} 秒）")
            break
        time.sleep(2)
    else:
        print("❌ ComfyUI 啟動逾時，請查看 /content/comfyui.log：")
        !tail -n 30 /content/comfyui.log
else:
    print(f"✅ ComfyUI 已經在 {COMFY_URL} 運行中")

# 同步 4 個遠端姿勢庫 + 本地 55 個精選姿勢 (共 136 個)
print("\n🕺 正在同步 VNCCS 136 個 3D 姿勢庫...")
pose_repos = [
    "Totemistyk/General_Poses_PoseStudio",
    "Totemistyk/Ballet_PoseStudio",
    "MIUProject/VNCCS_PoseLibrary_Main",
    "MIUProject/MySuperPoseRepo",
]
for repo in pose_repos:
    req = urllib.request.Request(
        f"{COMFY_URL}/vnccs/pose_library/repositories/add",
        data=json.dumps({"repo_id": repo}).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        urllib.request.urlopen(req, timeout=30)
    except Exception:
        pass  # 已存在的 repo 會回傳 400，可安全忽略

try:
    refresh_req = urllib.request.Request(
        f"{COMFY_URL}/vnccs/pose_library/repositories/refresh",
        data=b"{}",
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    urllib.request.urlopen(refresh_req, timeout=180)
except Exception as e:
    print(f"⚠️ 遠端姿勢庫刷新提示: {e}")

# 拷貝本包自帶的 55 個精選姿勢
src_local_poses = "/content/qwen21-nsfw-pose-skill/pose_library/local_user_poses"
dst_local_poses = "/content/ComfyUI/custom_nodes/ComfyUI_VNCCS_Utils/PoseLibrary/local_user_poses"
os.makedirs(dst_local_poses, exist_ok=True)
shutil.copytree(src_local_poses, dst_local_poses, dirs_exist_ok=True)

print("\n🩺 執行環境自檢 (scripts/doctor.py)：")
!python3 /content/qwen21-nsfw-pose-skill/scripts/doctor.py

In [ ]:
#@title 5. 【路線 ①】T2I 文字生成圖片（作者案例復刻 / 自訂提示詞）
#@markdown 選擇 `PRESET_CASE` 可直接復刻作者原圖配方，或選擇 `custom` 輸入自訂英文提示詞（建議依循 `SKILL.md` 的 7 要素結構，300~1500 字元）。

PRESET_CASE = "author_shower" #@param ["author_shower", "author_massage", "author_selfie", "custom"]
CUSTOM_PROMPT = "A hyper-realistic, low-angle shot of a young woman (23 years old) crouching by a sunlit indoor pool, wet dark hair clinging to her shoulders..." #@param {type:"string"}
NEGATIVE_PROMPT = "" #@param {type:"string"}
LORA_NAME = "qwen2_1-femaledVaginus-nsfw-v1_3-fine_L22-31.safetensors" #@param ["qwen2_1-femaledVaginus-nsfw-v1_3-fine_L22-31.safetensors", "qwen2_1-femaledVaginus-nsfw-v1.3-fine.safetensors", "thesealpacas-nsfw-v2.safetensors", "nsfw-posture-v3.safetensors"]
LORA_STRENGTH = 1.0 #@param {type:"slider", min:0.0, max:1.5, step:0.05}
WIDTH = 1152 #@param {type:"integer"}
HEIGHT = 1728 #@param {type:"integer"}
STEPS = 50 #@param {type:"integer"}
CFG = 2.5 #@param {type:"number"}
SAMPLER = "er_sde" #@param ["er_sde", "euler", "dpmpp_2m"]
SCHEDULER = "beta" #@param ["beta", "simple", "normal", "karras"]
SEED = 80964 #@param {type:"integer"}
OUTPUT_FILENAME = "route1_output.png" #@param {type:"string"}

import subprocess
from pathlib import Path
from IPython.display import Image, display

skill_dir = Path("/content/qwen21-nsfw-pose-skill")
out_path = skill_dir / "outputs" / OUTPUT_FILENAME
out_path.parent.mkdir(parents=True, exist_ok=True)

prompt_text = CUSTOM_PROMPT
neg_text = NEGATIVE_PROMPT
steps_val = STEPS
cfg_val = CFG
seed_val = SEED
lora_val = LORA_NAME
lora_str_val = LORA_STRENGTH

if PRESET_CASE == "author_shower":
    prompt_text = (skill_dir / "prompts/taws_v13_shower.txt").read_text().strip()
    steps_val, cfg_val, seed_val = 50, 2.5, 80964
elif PRESET_CASE == "author_massage":
    prompt_text = (skill_dir / "prompts/taws_v13_massage.txt").read_text().strip()
    steps_val, cfg_val, seed_val = 30, 2.5, 80964
elif PRESET_CASE == "author_selfie":
    raw = (skill_dir / "prompts/thesealpacas_v2_selfie.txt").read_text()
    parts = raw.split("[negative]")
    prompt_text = parts[0].strip()
    neg_text = parts[1].strip() if len(parts) > 1 else ""
    steps_val, cfg_val, seed_val = 50, 4.0, 124880844757561
    lora_val, lora_str_val = "thesealpacas-nsfw-v2.safetensors", 0.8

cmd = [
    "python3", str(skill_dir / "scripts/qwen_pose_test.py"),
    "--mode", "t2i",
    "--name", f"colab_{PRESET_CASE}",
    "--prompt", prompt_text,
    "--negative", neg_text,
    "--seed", str(seed_val),
    "--steps", str(steps_val),
    "--cfg", str(cfg_val),
    "--sampler", SAMPLER,
    "--scheduler", SCHEDULER,
    "--width", str(WIDTH),
    "--height", str(HEIGHT),
    "--lora", lora_val,
    "--lora-strength", str(lora_str_val),
    "--out", str(out_path),
]

print(f"🎨 開始執行路線 ① T2I ({WIDTH}x{HEIGHT}, {steps_val} steps, LoRA={lora_val}@{lora_str_val})...")
res = subprocess.run(cmd)
if res.returncode == 0 and out_path.exists():
    print(f"✅ 生成成功：{out_path}")
    display(Image(filename=str(out_path), width=480))
else:
    print("❌ 生成失敗，請檢查上方錯誤訊息或確認對應的 LoRA 已下載。")

In [ ]:
#@title 6. 【路線 ②】Edit 角色換姿勢 / 換裝（保留人物臉部與身份）
INPUT_IMAGE_PATH = "/content/qwen21-nsfw-pose-skill/references/src_standing.png" #@param {type:"string"}
EDIT_PROMPT = "change her pose to kneeling on one knee, hands on hips, keep face hair and outfit" #@param {type:"string"}
USE_NSFW_EDIT_LORA = False #@param {type:"boolean"}
EDIT_LORA_STRENGTH = 0.9 #@param {type:"slider", min:0.0, max:1.5, step:0.05}
STEPS = 20 #@param {type:"integer"}
CFG = 3.0 #@param {type:"number"}
SEED = 424242 #@param {type:"integer"}
OUTPUT_FILENAME = "route2_edit.png" #@param {type:"string"}

import subprocess
from pathlib import Path
from IPython.display import Image, display

skill_dir = Path("/content/qwen21-nsfw-pose-skill")
out_path = skill_dir / "outputs" / OUTPUT_FILENAME
out_path.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    "python3", str(skill_dir / "scripts/qwen_pose_test.py"),
    "--mode", "edit",
    "--name", "colab_edit",
    "--input", INPUT_IMAGE_PATH,
    "--prompt", EDIT_PROMPT,
    "--cfg", str(CFG),
    "--sampler", "er_sde",
    "--scheduler", "beta",
    "--steps", str(STEPS),
    "--seed", str(SEED),
    "--out", str(out_path),
]
if USE_NSFW_EDIT_LORA:
    cmd += ["--lora", "qwen21-nsfw-image-edit.safetensors", "--lora-strength", str(EDIT_LORA_STRENGTH)]

print(f"🪄 開始執行路線 ② Edit (input={INPUT_IMAGE_PATH})...")
res = subprocess.run(cmd)
if res.returncode == 0 and out_path.exists():
    print(f"✅ 編輯完成：{out_path}")
    display(Image(filename=str(out_path), width=480))

In [ ]:
#@title 7. 【路線 ③】腳本 3D 人偶姿勢遷移（免瀏覽器，需提供無貼圖 3D 人偶渲染圖）
#@markdown > 注意：`POSE_REF_IMAGE` 必須是**無五官/無貼圖的 3D 人偶渲染圖**（例如路線 ④ 輸出的 `doll_*.png`），不可使用真人照片或插畫。

CHARACTER_IMAGE = "/content/qwen21-nsfw-pose-skill/references/src_standing.png" #@param {type:"string"}
POSE_REF_IMAGE = "" #@param {type:"string"}
STEPS = 25 #@param {type:"integer"}
SEED = 424242 #@param {type:"integer"}
OUTPUT_FILENAME = "route3_pose_transfer.png" #@param {type:"string"}

import subprocess
from pathlib import Path
from IPython.display import Image, display

skill_dir = Path("/content/qwen21-nsfw-pose-skill")
out_path = skill_dir / "outputs" / OUTPUT_FILENAME
out_path.parent.mkdir(parents=True, exist_ok=True)

if not POSE_REF_IMAGE or not Path(POSE_REF_IMAGE).exists():
    print("⚠️ 請先填入有效的 3D 人偶姿勢參考圖路徑 (POSE_REF_IMAGE)，可先用下方【路線 ④】產生一張 doll_*.png！")
else:
    cmd = [
        "python3", str(skill_dir / "scripts/run_pose_transfer.py"),
        "--char", CHARACTER_IMAGE,
        "--pose-ref", POSE_REF_IMAGE,
        "--steps", str(STEPS),
        "--seed", str(SEED),
        "--out", str(out_path),
    ]
    print("🤖 開始執行路線 ③ 3D 人偶姿勢遷移...")
    res = subprocess.run(cmd)
    if res.returncode == 0 and out_path.exists():
        print(f"✅ 姿勢遷移完成：{out_path}")
        display(Image(filename=str(out_path), width=480))

In [ ]:
#@title 8. 【路線 ④】VNCCS PoseStudio — 從 136 個 3D 姿勢庫精確出圖（自動 Headless 瀏覽器）
ACTION = "run_pose" #@param ["list_poses", "run_pose"]
POSE_NAME = "squatting leaning back" #@param {type:"string"}
CHARACTER_IMAGE = "/content/qwen21-nsfw-pose-skill/references/src_standing.png" #@param {type:"string"}
SCENE_PROMPT = "on a rooftop at night, city lights bokeh" #@param {type:"string"}

import glob
import os
import subprocess
from pathlib import Path
from IPython.display import Image, display

skill_dir = Path("/content/qwen21-nsfw-pose-skill")

if ACTION == "list_poses":
    !python3 /content/qwen21-nsfw-pose-skill/scripts/run_vnccs_ui.py --list-poses
else:
    cmd = [
        "python3", str(skill_dir / "scripts/run_vnccs_ui.py"),
        "--pose", POSE_NAME,
        "--char", CHARACTER_IMAGE,
    ]
    if SCENE_PROMPT.strip():
        cmd += ["--prompt", SCENE_PROMPT.strip()]
    print(f"🎭 正在透過 Headless Chromium 執行 VNCCS PoseStudio (pose='{POSE_NAME}')...")
    res = subprocess.run(cmd)
    if res.returncode == 0:
        out_dir = Path("/content/qwen21-nsfw-pose-skill/outputs/vnccs_ui")
        pngs = sorted(out_dir.glob("*.png"), key=os.path.getmtime, reverse=True)
        if pngs:
            print(f"✅ 最新輸出檔案：{pngs[0]}")
            display(Image(filename=str(pngs[0]), width=480))

In [ ]:
#@title 9. （選用）開啟 Cloudflare Tunnel 進入 ComfyUI 網頁介面手動調整 3D 人偶
#@markdown 如果你想在瀏覽器裡親自拖曳旋轉 `VNCCS_PoseStudio` 的 3D 人偶骨骼，執行此區塊會產生一個 `https://xxxx.trycloudflare.com` 網址。

import re
import subprocess
import time

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

tunnel_log = open("/content/cloudflared.log", "w")
proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

print("⏳ 正在建立 Cloudflare Tunnel...")
for _ in range(30):
    time.sleep(1)
    content = open("/content/cloudflared.log").read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
    if m:
        print(f"\n🌐 ComfyUI 外部存取網址： {m.group(0)}")
        print("（打開網頁後，可載入 /content/qwen21-nsfw-pose-skill/workflows/VNCCS_PoseStudio_QI2.1.json 工作流）")
        break
else:
    print("⚠️ 未能取得網址，請檢查 /content/cloudflared.log")